# 01 · Understand and profile the source

Business question: where do delays and cancellations concentrate in the supplied 2015 US flight records? Each row is one source observation. This notebook verifies the original files and profiles all rows without loading the full CSV into memory. Run all cells, inspect the output, then continue to Notebook 02.

In [ ]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
assert (ROOT / "sql" / "01_setup.sql").is_file(), "Open Jupyter from the project root."
DATA = ROOT / "notebooks" / "data"
RAW = DATA / "raw"
OUT = DATA / "processed"
EXPORT = DATA / "exports"
QUALITY = DATA / "quality"
for folder in (OUT, EXPORT, QUALITY):
    folder.mkdir(parents=True, exist_ok=True)


## Verify provenance

The existing manifest is the source record. Stop if any CSV changes. Hash verification is not a new download or a new license review.

In [ ]:
manifest = json.loads((RAW / "manifest.json").read_text(encoding="utf-8"))
source_checks = []
for item in manifest["files"]:
    path = RAW / item["name"]
    with path.open("rb") as handle:
        actual_hash = hashlib.file_digest(handle, "sha256").hexdigest()
    assert actual_hash == item["sha256"], f"Source changed: {path.name}"
    assert path.stat().st_size == item["bytes"], f"Source size changed: {path.name}"
    source_checks.append({"file": path.name, "bytes": path.stat().st_size, "sha256": actual_hash})
display(pd.DataFrame(source_checks))
display(pd.read_csv(RAW / "airlines.csv"))
display(pd.read_csv(RAW / "airports.csv").head())


## Profile every source row

Missing arrival delay is not zero delay. Negative delay means early arrival. Numeric airport identifiers remain text. Check whether they occur in a particular month before making a route comparison.

In [ ]:
rows = 0
missing = None
monthly = []
ranges = []
for chunk in pd.read_csv(RAW / "flights.csv", chunksize=250_000, low_memory=False):
    rows += len(chunk)
    missing = chunk.isna().sum() if missing is None else missing.add(chunk.isna().sum())
    assert chunk["CANCELLED"].isin([0, 1]).all()
    assert chunk["DIVERTED"].isin([0, 1]).all()
    part = chunk.assign(
        numeric_origin=chunk["ORIGIN_AIRPORT"].astype("string").str.fullmatch(r"\d+"),
        numeric_destination=chunk["DESTINATION_AIRPORT"].astype("string").str.fullmatch(r"\d+"),
    ).groupby("MONTH").agg(
        scheduled_flights=("YEAR", "size"),
        numeric_origin_rows=("numeric_origin", "sum"),
        numeric_destination_rows=("numeric_destination", "sum"),
    )
    monthly.append(part)
    ranges.append(chunk[["ARRIVAL_DELAY", "DEPARTURE_DELAY"]].agg(["min", "max"]))

source_columns = pd.read_csv(RAW / "flights.csv", nrows=0).columns.tolist()
null_profile = pd.DataFrame({"missing_rows": missing, "missing_fraction": missing / rows})
airport_profile = pd.concat(monthly).groupby(level=0).sum()
assert rows == 5_819_079 and len(source_columns) == 31, "Unexpected supplied-source size."
display(null_profile)
display(airport_profile)
airport_profile.to_csv(QUALITY / "airport_code_profile.csv")
report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "row_count": rows, "column_count": len(source_columns),
    "columns": source_columns, "null_counts": {k: int(v) for k, v in missing.items()},
    "arrival_delay_min": float(min(frame.loc["min", "ARRIVAL_DELAY"] for frame in ranges)),
    "arrival_delay_max": float(max(frame.loc["max", "ARRIVAL_DELAY"] for frame in ranges)),
    "source_files": source_checks,
}
(QUALITY / "data_quality_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Verified {rows:,} rows and {len(source_columns)} columns.")


## Interpretation to write yourself

Explain why a cancellation stays in the fact, why a missing delay stays missing, and why unresolved numeric airport identifiers cannot be guessed from the IATA lookup. Read `reference/metric_dictionary.csv`. The project retains its existing custom threshold: on time is **≤15 minutes**, late is **>15**, severe is **>60**. BTS officially defines delayed as **≥15**, so do not call this project's on-time percentage the official BTS rate.

Next: run `02_clean_transform.ipynb`.